# Fase 3 — Correzione della serie storica e indice di gentrificazione

Questo notebook riprende il data warehouse prodotto dalla Fase 2 e corregge
il difetto emerso dall'analisi dei risultati Athena: una **rottura di
composizione del campione nel 2014**.

## Il problema in una tabella

| anni | zone osservate | media EUR/m2 |
|---|---|---|
| 2004-2013 | 742-750 | 2.715 → 2.964 |
| **2014** | **340** (−55%) | **3.820 (+28,9%)** |
| 2015-2023 | 344-350 | 3.788 → 4.716 |

Nel 2014 il dataset perde oltre meta delle zone e nello stesso istante la
media salta del 29%. Non e il mercato — il 2014 e vicino al minimo della
crisi immobiliare italiana, e in tutti gli altri anni le variazioni stanno
fra −1,7% e +8,9%. E la **composizione** del campione che cambia: escono le
zone economiche e restano le care.

La crescita apparente 2004→2024 e del +89%. Ma dentro il primo periodo e
+9% e dentro il secondo +34%: concatenandoli si ottiene +46%. I restanti
**43 punti sono artefatto**, non rivalutazione.

Questo invalida ogni confronto che attraversi il 2014 — cioe esattamente il
confronto 2004-08 contro 2020+ su cui poggiava l'indice di gentrificazione.

## Che cosa fa questo notebook

1. documenta la rottura e ne misura l'entita
2. costruisce un **indice concatenato a panel appaiato**, immune ai cambi
   di composizione
3. ricalcola la crescita di ogni zona **depurata dall'andamento cittadino**
4. aggrega ai quartieri e ricostruisce l'indice di gentrificazione
5. esporta i risultati corretti

## 1. Setup

In [ ]:
# !pip install -q duckdb pandas matplotlib
import json
import duckdb
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path

for _s in ('seaborn-v0_8-whitegrid', 'seaborn-whitegrid', 'ggplot'):
    if _s in plt.style.available:
        plt.style.use(_s)
        break
pd.set_option('display.max_columns', None)
pd.set_option('display.width', 140)

# Cartella di lavoro: la stessa delle fasi precedenti.
# In Colab con Drive montato:
#   os.environ['MILANO_BASE'] = '/content/drive/MyDrive/milano-realestate'
import os
BASE = Path(os.environ.get('MILANO_BASE', 'data/milano-realestate'))
OUT = BASE / 'output'
OUT.mkdir(parents=True, exist_ok=True)

DW = BASE / 'milano_dw.duckdb'
if not DW.exists():
    raise FileNotFoundError(
        f'Warehouse non trovato in {DW}. Esegui prima la Fase 2, oppure '
        'imposta MILANO_BASE sul percorso giusto.')

con = duckdb.connect(str(DW))
tabelle = [r[0] for r in con.execute('SHOW TABLES').fetchall()]
print(f'Warehouse: {DW}')
print(f'Tabelle: {tabelle}')

for t in ('fact_property_prices', 'bridge_geography'):
    if t not in tabelle:
        print(f'ATTENZIONE: manca {t}')

# Parametri dell'analisi, raccolti qui perche vanno dichiarati nell'elaborato
MIN_ZONE_APPAIATE = 20    # zone comuni minime per accettare un anello annuale
MIN_DURATA = 5            # anni minimi di serie per includere una zona
MIN_ANNUNCI_BNB = 50      # annunci minimi perche il tasso per 1.000 ab. sia stabile

## 2. La rottura del 2014, misurata

Prima di correggere, si documenta. Il numero di zone osservate per anno e la
spia: se cambia bruscamente, ogni confronto fra livelli attraversa una
discontinuita.

In [ ]:
# Prezzi per zona-anno. Questa aggregazione fa gia due cose utili:
#  - collassa le righe duplicate del semestre 2024/2, che nella Fase 2
#    sfuggivano alla deduplica perche differivano per una colonna
#  - porta il dato alla grana su cui si costruisce l'indice
za = con.execute("""
    SELECT zone_code, year, AVG(price_per_m2) AS prezzo, COUNT(*) AS oss
    FROM   fact_property_prices
    WHERE  price_per_m2 IS NOT NULL
    GROUP  BY zone_code, year
""").df().sort_values(['zone_code', 'year'])

print(f'Serie zona-anno: {len(za):,} righe | {za.zone_code.nunique()} zone | '
      f'{za.year.min():.0f}-{za.year.max():.0f}')

conteggio = (za.groupby('year')
             .agg(zone=('zone_code', 'nunique'), oss=('oss', 'sum'),
                  media=('prezzo', 'mean'))
             .reset_index())
conteggio['var_zone_pct'] = conteggio['zone'].pct_change() * 100
conteggio['var_media_pct'] = conteggio['media'].pct_change() * 100
print()
print(conteggio.round(1).to_string(index=False))

sospetti = conteggio[conteggio['var_zone_pct'].abs() > 20]
print('\nAnni con variazione del campione superiore al 20%:')
print(sospetti[['year', 'zone', 'var_zone_pct', 'media', 'var_media_pct']]
      .round(1).to_string(index=False) if len(sospetti) else '  nessuno')

In [ ]:
fig, ax1 = plt.subplots(figsize=(12, 5))
ax1.bar(conteggio['year'], conteggio['zone'], color='#C9D6CE',
        label='zone osservate')
ax1.set_ylabel('Zone osservate'); ax1.set_xlabel('Anno')
ax2 = ax1.twinx()
ax2.plot(conteggio['year'], conteggio['media'], color='#B85042',
         marker='o', linewidth=2, label='media EUR/m2')
ax2.set_ylabel('Media EUR/m2')
ax1.set_title('Il salto della media coincide con il crollo del campione')
for a in sospetti['year']:
    ax1.axvline(a, color='#B85042', linestyle='--', alpha=0.5)
fig.legend(loc='upper left', bbox_to_anchor=(0.12, 0.88))
plt.tight_layout()
plt.savefig(OUT / 'rottura_composizione.png', dpi=150, bbox_inches='tight')
plt.show()
print('Dove la barra crolla e la linea salta nello stesso anno, la variazione '
      'di prezzo non e di mercato ma di composizione.')

## 3. Indice concatenato a panel appaiato

Il principio: **non si confronta mai il livello di due anni lontani**. Si
confronta ogni anno con il precedente usando **solo le zone presenti in
entrambi**, e poi si moltiplicano le variazioni una dopo l'altra.

Se una zona esce dal campione, sparisce da entrambi i lati del rapporto e non
sposta l'indice. È il metodo con cui si costruiscono gli indici dei prezzi
ufficiali, per lo stesso motivo.

Per l'anello annuale si usa la **mediana** dei rapporti e non la media: e
robusta alle singole zone con variazioni anomale.

In [ ]:
anni = sorted(za.year.unique())
anelli = []
for t0, t1 in zip(anni[:-1], anni[1:]):
    a = za[za.year == t0].set_index('zone_code')['prezzo']
    b = za[za.year == t1].set_index('zone_code')['prezzo']
    comuni = a.index.intersection(b.index)
    riga = {'anno': t1, 'zone_appaiate': len(comuni),
            'zone_uscite': len(a.index.difference(b.index)),
            'zone_entrate': len(b.index.difference(a.index))}
    if len(comuni) < MIN_ZONE_APPAIATE:
        riga['link'] = np.nan
    else:
        rap = (b[comuni] / a[comuni]).replace([np.inf, -np.inf], np.nan).dropna()
        riga['link'] = float(rap.median())
    anelli.append(riga)

anelli = pd.DataFrame(anelli)
anelli['var_pct'] = (anelli['link'] - 1) * 100

# Concatenamento: base 100 al primo anno
indice = {anni[0]: 100.0}
for _, r in anelli.iterrows():
    prec = indice.get(r['anno'] - 1, np.nan)
    indice[r['anno']] = prec * r['link'] if pd.notna(r['link']) else np.nan
idx = pd.DataFrame({'year': list(indice), 'indice_concatenato': list(indice.values())})

# Confronto con l'indice ingenuo, costruito sui livelli medi
ingenuo = za.groupby('year')['prezzo'].mean().reset_index(name='media_grezza')
ingenuo['indice_ingenuo'] = ingenuo['media_grezza'] / ingenuo['media_grezza'].iloc[0] * 100

conf = (idx.merge(ingenuo, on='year')
        .merge(anelli[['anno', 'zone_appaiate', 'zone_uscite', 'var_pct']],
               left_on='year', right_on='anno', how='left').drop(columns='anno'))
print(conf[['year', 'zone_appaiate', 'zone_uscite', 'var_pct',
            'indice_concatenato', 'indice_ingenuo']].round(1).to_string(index=False))

In [ ]:
print('=' * 70)
print('DOVE IL CAMPIONE CAMBIA COMPOSIZIONE')
print('=' * 70)
grezza = ingenuo.set_index('year')['media_grezza']
rotture = anelli[anelli['zone_uscite'] > anelli['zone_appaiate'] * 0.2]
if len(rotture):
    for _, r in rotture.iterrows():
        vg = (grezza[r['anno']] / grezza[r['anno'] - 1] - 1) * 100
        print(f"  {r['anno']:.0f}: escono {r['zone_uscite']:.0f} zone su "
              f"{r['zone_appaiate'] + r['zone_uscite']:.0f}")
        print(f"       media grezza {vg:+.1f}%  ->  panel appaiato "
              f"{r['var_pct']:+.1f}%   (artefatto: {vg - r['var_pct']:+.1f} punti)")
else:
    print('  Nessun anno con uscite superiori al 20%.')

c_ing = conf['indice_ingenuo'].iloc[-1] / 100 - 1
c_cat = conf['indice_concatenato'].iloc[-1] / 100 - 1
print(f"\nCrescita {anni[0]:.0f}-{anni[-1]:.0f}:")
print(f"  letta sulla media grezza  : {c_ing:+.0%}")
print(f"  indice concatenato        : {c_cat:+.0%}")
print(f"  sovrastima da composizione: {(c_ing - c_cat) * 100:+.0f} punti")

plt.figure(figsize=(12, 5))
plt.plot(conf['year'], conf['indice_ingenuo'], marker='s', linestyle='--',
         color='#A9A29C', linewidth=2, label='indice ingenuo (livelli medi)')
plt.plot(conf['year'], conf['indice_concatenato'], marker='o',
         color='#B85042', linewidth=2.5, label='indice concatenato')
for a in rotture['anno']:
    plt.axvline(a, color='#B85042', linestyle=':', alpha=0.6)
plt.title('La distanza fra le due curve e l\'artefatto di composizione')
plt.xlabel('Anno'); plt.ylabel('Indice, base 100')
plt.legend(); plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig(OUT / 'indice_concatenato.png', dpi=150, bbox_inches='tight')
plt.show()

## 4. Crescita per zona, depurata dall'andamento cittadino

Per ogni zona si confronta il proprio **primo e ultimo triennio** disponibile,
e si divide per quanto ha fatto la citta **negli stessi anni**.

Così finestre di lunghezza diversa restano confrontabili: si misura il
**sorpasso sulla citta**, non il livello assoluto. Un quartiere che cresce
quanto Milano non si sta trasformando — sta seguendo il mercato.

In [ ]:
idx_map = idx.set_index('year')['indice_concatenato']


def crescita_zona(g):
    g = g.sort_values('year')
    aa = g['year'].unique()
    if len(aa) < MIN_DURATA:
        return None
    primi, ultimi = aa[:3], aa[-3:]          # trienni: attenuano il rumore
    p0 = g.loc[g.year.isin(primi), 'prezzo'].mean()
    p1 = g.loc[g.year.isin(ultimi), 'prezzo'].mean()
    i0 = idx_map.reindex(primi).mean()
    i1 = idx_map.reindex(ultimi).mean()
    if not (p0 > 0) or pd.isna(i0) or pd.isna(i1) or i0 <= 0:
        return None
    durata = ultimi.max() - primi.min()
    sorpasso = (p1 / p0) / (i1 / i0)         # > 1 = ha battuto la citta
    return pd.Series({'anno_inizio': primi.min(), 'anno_fine': ultimi.max(),
                      'durata': durata, 'prezzo_inizio': p0, 'prezzo_fine': p1,
                      'crescita_nominale': p1 / p0 - 1,
                      'sorpasso_citta': sorpasso - 1,
                      'sorpasso_annuo': sorpasso ** (1 / durata) - 1})


per_zona = (za.groupby('zone_code', group_keys=False)
            .apply(crescita_zona, include_groups=False)
            .dropna(how='all').reset_index())
print(f'Zone con serie >= {MIN_DURATA} anni: {len(per_zona)} '
      f'su {za.zone_code.nunique()}')
print(f'Finestre osservate: {per_zona.anno_inizio.min():.0f}-'
      f'{per_zona.anno_fine.max():.0f}, durata mediana '
      f'{per_zona.durata.median():.0f} anni')

# Controllo: la depurazione ha neutralizzato il vantaggio di chi entra tardi?
if per_zona['anno_inizio'].nunique() > 1:
    r_nom = per_zona[['anno_inizio', 'crescita_nominale']].corr().iloc[0, 1]
    r_sor = per_zona[['anno_inizio', 'sorpasso_annuo']].corr().iloc[0, 1]
    print(f'\nCorrelazione fra anno di inizio serie e crescita misurata:')
    print(f'  crescita nominale : {r_nom:+.3f}')
    print(f'  sorpasso depurato : {r_sor:+.3f}')
    print('  Piu vicino a zero significa misura indipendente dalla finestra.')
    if abs(r_sor) >= abs(r_nom):
        print('  ATTENZIONE: la distorsione non si e ridotta. Limita '
              'l\'analisi alle serie lunghe e dichiaralo.')

## 5. Aggregazione ai quartieri

In [ ]:
bridge = con.execute("""
    SELECT zone_code, nil_norm FROM bridge_geography WHERE nil_norm IS NOT NULL
""").df()
print(f'Raccordo: {len(bridge)} zone -> {bridge.nil_norm.nunique()} quartieri')

pz = per_zona.merge(bridge, on='zone_code', how='inner')
print(f'Zone con crescita calcolata E raccordate: {len(pz)}')

nil = (pz.groupby('nil_norm')
       .agg(zone=('zone_code', 'nunique'),
            anno_inizio=('anno_inizio', 'min'), anno_fine=('anno_fine', 'max'),
            prezzo_inizio=('prezzo_inizio', 'mean'),
            prezzo_fine=('prezzo_fine', 'mean'),
            crescita_nominale=('crescita_nominale', 'mean'),
            sorpasso_citta=('sorpasso_citta', 'mean'),
            sorpasso_annuo=('sorpasso_annuo', 'mean'))
       .reset_index().sort_values('sorpasso_annuo', ascending=False))

m = nil.head(20).copy()
for col in ('crescita_nominale', 'sorpasso_citta', 'sorpasso_annuo'):
    m[col] = (m[col] * 100).round(1)
m['prezzo_inizio'] = m['prezzo_inizio'].round(0)
m['prezzo_fine'] = m['prezzo_fine'].round(0)
print()
print(m.to_string(index=False))
print('\n  crescita_nominale: variazione del prezzo, include il mercato')
print('  sorpasso_citta   : quanto ha fatto MEGLIO della citta, in punti %')
print('  sorpasso_annuo   : lo stesso su base annua, confrontabile fra serie')
print('                     di lunghezza diversa')
print(f'\nCopertura: {len(nil)} quartieri')

## 6. Indice di gentrificazione ricalcolato

Rispetto alla versione precedente cambiano due cose:

- la componente immobiliare usa il **sorpasso annuo depurato** invece della
  crescita grezza, quindi non e piu contaminata dalla rottura del 2014
- il dato Airbnb viene usato solo dove poggia su un numero sufficiente di
  annunci: un tasso per mille abitanti calcolato su due annunci non e una
  misura, e rumore

Resta il **filtro di ammissibilita**: solo i quartieri che non erano gia fra
i piu cari all'inizio della propria serie. Senza, l'indice classifica il
centro storico come gentrificato.

In [ ]:
# --- Airbnb per quartiere, con soglia minima ---
try:
    bnb = con.execute(f"""
        SELECT n.nil_norm, COUNT(*) AS annunci,
               SUM(CASE WHEN r.is_entire_home THEN 1 ELSE 0 END) AS interi,
               AVG(CASE WHEN r.is_multi_host THEN 1.0 ELSE 0.0 END) AS quota_professionali
        FROM   fact_short_rentals r
        JOIN   dim_neighbourhood n ON r.nil_id = n.nil_id
        WHERE  n.nil_norm IS NOT NULL
        GROUP  BY n.nil_norm
        HAVING COUNT(*) >= {MIN_ANNUNCI_BNB}
    """).df()
    print(f'Quartieri con almeno {MIN_ANNUNCI_BNB} annunci Airbnb: {len(bnb)}')
except Exception as e:
    bnb = pd.DataFrame(columns=['nil_norm', 'annunci', 'interi', 'quota_professionali'])
    print(f'Dati Airbnb non disponibili: {str(e)[:90]}')

# --- Demografia per quartiere ---
demo = con.execute("""
    SELECT UPPER(TRIM(zone_name)) AS nil_norm,
           MAX(CASE WHEN year <= 2013 THEN population END) AS pop_ini,
           MAX(CASE WHEN year >= 2022 THEN population END) AS pop_fin,
           MAX(CASE WHEN year <= 2013 THEN foreigners END) AS stra_ini,
           MAX(CASE WHEN year >= 2022 THEN foreigners END) AS stra_fin
    FROM   dim_demographics WHERE zone_name IS NOT NULL
    GROUP  BY UPPER(TRIM(zone_name))
""").df()
print(f'Quartieri con demografia: {len(demo)}')

g = nil.merge(bnb, on='nil_norm', how='left').merge(demo, on='nil_norm', how='left')
g['var_popolazione'] = (g['pop_fin'] - g['pop_ini']) / g['pop_ini']
g['quota_stra_ini'] = g['stra_ini'] / g['pop_ini']
g['quota_stra_fin'] = g['stra_fin'] / g['pop_fin']
g['var_quota_stranieri'] = g['quota_stra_fin'] - g['quota_stra_ini']
g['airbnb_interi_per_1000'] = np.where(
    g['pop_fin'] > 0, g['interi'] / g['pop_fin'] * 1000, np.nan)

print(f'\nQuartieri nel calcolo finale: {len(g)}')
for col, et in [('sorpasso_annuo', 'sorpasso immobiliare'),
                ('airbnb_interi_per_1000', 'pressione turistica'),
                ('var_quota_stranieri', 'ricambio demografico')]:
    print(f'  {et:24s} disponibile su {g[col].notna().sum():>3}/{len(g)}')

In [ ]:
# --- Filtro di ammissibilita ---
QUANTILE_SOGLIA = 0.75
soglia = g['prezzo_inizio'].quantile(QUANTILE_SOGLIA)
g['gentrificabile'] = g['prezzo_inizio'] < soglia
print(f'Soglia (quantile {QUANTILE_SOGLIA:.0%} del prezzo iniziale): '
      f'{soglia:,.0f} EUR/m2')
print(f'Quartieri ammissibili: {g["gentrificabile"].sum()}/{len(g)}')
print(f'  esclusi: {sorted(g.loc[~g["gentrificabile"], "nil_norm"])[:10]}')


def z(s):
    s = pd.to_numeric(s, errors='coerce')
    sd = s.std()
    return (s - s.mean()) / sd if sd and sd > 0 else s * 0


COMPONENTI = {
    'sorpasso_annuo':         ('Sorpasso immobiliare', +1, 0.45),
    'airbnb_interi_per_1000': ('Pressione turistica',  +1, 0.30),
    'var_quota_stranieri':    ('Ricambio demografico', -1, 0.25),
}

# I pesi vengono rinormalizzati RIGA PER RIGA sulle componenti effettivamente
# disponibili. Riempire i mancanti con zero equivarrebbe ad assegnare un
# valore medio a un dato che non c'e.
for col, (_, segno, _) in COMPONENTI.items():
    g[f'z_{col}'] = z(g[col]) * segno if col in g.columns else np.nan


def indice_riga(r):
    num = den = 0.0
    for col, (_, _, peso) in COMPONENTI.items():
        v = r.get(f'z_{col}')
        if pd.notna(v):
            num += v * peso
            den += peso
    return num / den if den > 0 else np.nan


g['indice'] = g.apply(indice_riga, axis=1)
g['componenti_usate'] = g.apply(
    lambda r: sum(pd.notna(r.get(f'z_{c}')) for c in COMPONENTI), axis=1)

q = g['indice'].quantile([0.5, 0.75])


def classifica(r):
    if pd.isna(r['indice']):
        return 'Dati insufficienti'
    if not r['gentrificabile']:
        return ('Rivalutazione di area gia affluente' if r['indice'] >= q[0.5]
                else 'Area affluente stabile')
    if r['indice'] >= q[0.75]:
        return 'Gentrificazione avanzata'
    if r['indice'] >= q[0.5]:
        return 'Gentrificazione in corso'
    return 'Area popolare stabile'


g['classe'] = g.apply(classifica, axis=1)
g = g.sort_values('indice', ascending=False)

print('\n' + '=' * 70)
print('DISTRIBUZIONE')
print('=' * 70)
print(g['classe'].value_counts().to_string())

col_out = ['nil_norm', 'anno_inizio', 'anno_fine', 'prezzo_inizio', 'prezzo_fine',
           'crescita_nominale', 'sorpasso_annuo', 'airbnb_interi_per_1000',
           'var_popolazione', 'var_quota_stranieri', 'componenti_usate',
           'indice', 'classe']
col_out = [c for c in col_out if c in g.columns]

print('\n' + '=' * 70)
print('QUARTIERI AMMISSIBILI, ordinati per indice')
print('=' * 70)
print(g[g['gentrificabile']].head(15)[col_out].round(3).to_string(index=False))

print('\n' + '=' * 70)
print('CONTROLLO: aree gia care all\'inizio (NON gentrificazione)')
print('=' * 70)
print(g[~g['gentrificabile']].head(8)[col_out].round(3).to_string(index=False))

## 7. Verifiche di robustezza

I pesi e la soglia sono scelte dell'analista, non dati. Se la graduatoria
regge al variare di entrambi, la conclusione non e un artefatto della
parametrizzazione — e questo va scritto nell'elaborato.

In [ ]:
colonne_z = [f'z_{c}' for c in COMPONENTI]
disponibili = [c for c in colonne_z if c in g.columns]

alternative = {
    'base (45/30/25)':       (0.45, 0.30, 0.25),
    'solo immobiliare':      (1.00, 0.00, 0.00),
    'pesi uguali':           (0.34, 0.33, 0.33),
    'turismo dominante':     (0.25, 0.55, 0.20),
}
classifiche = {}
for nome, pesi in alternative.items():
    pesi_u = [p for c, p in zip(colonne_z, pesi) if c in g.columns]
    if not pesi_u or sum(pesi_u) == 0:
        continue
    punteggio = sum(g[c].fillna(0) * p for c, p in zip(disponibili, pesi_u)) / sum(pesi_u)
    classifiche[nome] = punteggio.rank(ascending=False)

print('SENSIBILITA AI PESI')
if len(classifiche) > 1:
    rank_df = pd.DataFrame(classifiche)
    base = list(classifiche)[0]
    for nome in list(classifiche)[1:]:
        rho = rank_df[base].corr(rank_df[nome], method='spearman')
        print(f'  {nome:22s} rho di Spearman = {rho:+.3f}')
    top5 = set(g.head(5)['nil_norm'])
    print(f'\n  Top 5 base: {sorted(top5)}')
    for nome in list(classifiche)[1:]:
        alt = set(g.assign(r=classifiche[nome]).nsmallest(5, 'r')['nil_norm'])
        print(f'    stabili con "{nome}": {len(top5 & alt)}/5')

print('\nSENSIBILITA ALLA SOGLIA DI AMMISSIBILITA')
for qs in (0.50, 0.66, 0.75, 0.85):
    s_alt = g['prezzo_inizio'].quantile(qs)
    n = (g['prezzo_inizio'] < s_alt).sum()
    entrano = (set(g.loc[g['prezzo_inizio'] < s_alt, 'nil_norm']) -
               set(g.loc[g['gentrificabile'], 'nil_norm']))
    print(f'  quantile {qs:.0%} ({s_alt:,.0f} EUR/m2): {n:>3} ammissibili'
          + (f'  in piu: {sorted(entrano)[:3]}' if entrano else ''))

print('\nSENSIBILITA ALLA LUNGHEZZA MINIMA DELLA SERIE')
for m_ in (4, 6, 8, 10, 12):
    n = per_zona[per_zona['durata'] >= m_ - 1].merge(
        bridge, on='zone_code')['nil_norm'].nunique()
    print(f'  serie >= {m_:>2} anni: {n:>3} quartieri')

In [ ]:
# --- Grafico di sintesi ---
d = g[g['indice'].notna()].copy()
if len(d):
    fig, ax = plt.subplots(figsize=(12, 7))
    colori = {'Gentrificazione avanzata': '#B85042',
              'Gentrificazione in corso': '#C8776A',
              'Rivalutazione di area gia affluente': '#A7BEAE',
              'Area affluente stabile': '#C9D6CE',
              'Area popolare stabile': '#B9BFC4'}
    for cl, sub in d.groupby('classe'):
        ax.scatter(sub['prezzo_inizio'], sub['sorpasso_annuo'] * 100,
                   s=90, alpha=0.85, label=cl,
                   color=colori.get(cl, '#999999'), edgecolors='white')
    ax.axvline(soglia, color='#B85042', linestyle='--', alpha=0.7)
    ax.text(soglia, ax.get_ylim()[1] * 0.95, ' soglia di ammissibilita',
            color='#B85042', fontsize=9, va='top')
    ax.axhline(0, color='grey', linewidth=1, alpha=0.5)
    for _, r in d.nlargest(8, 'sorpasso_annuo').iterrows():
        ax.annotate(str(r['nil_norm'])[:22],
                    (r['prezzo_inizio'], r['sorpasso_annuo'] * 100),
                    fontsize=8, alpha=0.85,
                    xytext=(4, 4), textcoords='offset points')
    ax.set_xlabel('Prezzo a inizio serie (EUR/m2)')
    ax.set_ylabel('Sorpasso annuo sulla citta (punti %)')
    ax.set_title('Chi parte basso e supera la citta: il quadrante in alto a sinistra')
    ax.legend(fontsize=9, loc='upper right')
    ax.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.savefig(OUT / 'gentrificazione_quadranti.png', dpi=150, bbox_inches='tight')
    plt.show()
    print('Il quadrante in alto a sinistra — prezzo iniziale basso, sorpasso '
          'positivo — e il profilo della gentrificazione.')
else:
    print('Nessun quartiere con indice calcolabile.')

## 8. Esportazione

In [ ]:
g[col_out].to_csv(OUT / 'indice_gentrificazione_corretto.csv', index=False)
nil.to_csv(OUT / 'crescita_depurata_nil.csv', index=False)
idx.to_csv(OUT / 'indice_concatenato_citta.csv', index=False)
conf.to_csv(OUT / 'confronto_indici.csv', index=False)

print('File prodotti:')
for f in sorted(OUT.glob('*.csv')):
    print(f'  {f.name:42s} {f.stat().st_size:>9,} bytes')
for f in sorted(OUT.glob('*.png')):
    print(f'  {f.name:42s} {f.stat().st_size:>9,} bytes')

print(f"""
=====================================================================
SINTESI DA RIPORTARE NELL'ELABORATO
=====================================================================
Rottura di composizione : {len(rotture)} anni con uscite oltre il 20% del campione
Crescita grezza         : {c_ing:+.0%}
Crescita depurata       : {c_cat:+.0%}
Artefatto di composizione: {(c_ing - c_cat) * 100:+.0f} punti
Quartieri analizzati    : {len(g)} (metodo a finestre fisse: 13)
Di cui ammissibili      : {g['gentrificabile'].sum()}
Parametri dichiarati    : serie >= {MIN_DURATA} anni, soglia al
                          {QUANTILE_SOGLIA:.0%}, Airbnb >= {MIN_ANNUNCI_BNB} annunci
=====================================================================""")

## Limiti che restano

**La rottura del 2014 e corretta, non eliminata.** Le zone uscite dal
campione non tornano: per loro la serie si ferma al 2013 e il sorpasso e
misurato su una finestra piu corta. Il dato annualizzato lo rende
confrontabile, ma con piu incertezza.

**Il raccordo zone OMI ↔ NIL resta approssimato.** Le zone a cavallo di piu
quartieri sono assegnate a quello di massima sovrapposizione: e una scelta,
e la colonna `quota` di `bridge_geography` ne misura l'affidabilita.

**L'indice e un costrutto.** Le tre componenti e i loro pesi sono scelte
motivate ma non uniche. Le verifiche della sezione 7 servono a mostrare che
la graduatoria non dipende in modo critico da quelle scelte — non a
dimostrare che siano le uniche possibili.

**Nessuna di queste misure e causale.** Sono associazioni fra aggregati
territoriali. Dedurne comportamenti individuali sarebbe una fallacia
ecologica, ed e l'obiezione piu facile da muovere a un lavoro di questo
tipo.